# 실습 5: 데이터 전처리 + 특성 추출

HTTP 요청 텍스트에서 URL 디코딩, 텍스트 정리를 수행하고,
분류에 유용한 숫자형 특성(Feature)을 추출합니다.

**실행 방법**: JupyterLab에서 셀을 위에서부터 `Shift+Enter`로 순차 실행

**사전 준비**: `python download_csic2010.py`로 `csic2010_requests.csv` 생성

**다음 단계**: `feature_engineering.ipynb`

In [1]:
import pandas as pd
import numpy as np
import os
import re
import math
from urllib.parse import unquote, urlparse, parse_qs

print("=" * 65)
print("  [실습 5] 데이터 전처리 + 특성 추출")
print("=" * 65)

# 데이터 로드
# Jupyter / 스크립트 양쪽에서 동작하도록 SCRIPT_DIR을 결정
try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    SCRIPT_DIR = os.getcwd()

data_path = os.path.join(SCRIPT_DIR, "csic2010_requests.csv")
if not os.path.exists(data_path):
    print(f"\n  !! csic2010_requests.csv 파일이 없습니다.")
    print(f"  >> 먼저 download_csic2010.py를 실행하세요.")
    raise SystemExit('필수 파일이 없어 중단합니다')

df = pd.read_csv(data_path)
print(f"\n  원본 데이터: {df.shape[0]:,}행 x {df.shape[1]}열")

  [실습 5] 데이터 전처리 + 특성 추출

  원본 데이터: 10,000행 x 18열


## Step 0 (선택): GET 요청만 + 중복 제거 + 10,000건 재샘플링

기본 `csic2010_requests.csv`는 GET/POST/PUT이 섞여 있고 (method, url, body) 기준 중복도 51%(5,127건)에 달합니다. **GET만 + 고유 패턴 1만 건**으로 분석 범위를 정리하려면 아래 셀을 실행하세요:

1. Kaggle 원본 `csic_database.csv`를 다시 로드 (61,065건)
2. `download_csic2010.py`와 동일한 컬럼 정규화 적용
3. **`method == "GET"`으로 필터링** ★
4. `(method, url, body)` 기준 **중복 제거**
5. 라벨 비율 유지하며 **최대 10,000건 재샘플링**
6. `df`를 새 버전으로 교체 + `csic2010_get_unique.csv`로 저장

> **GET만 사용하는 이유**:
> - URL 쿼리스트링에 모든 페이로드가 노출되어 분석이 직관적
> - POST 본문 결측(`body=NaN`) 이슈에서 자유로움
> - 단, **POST 기반 공격(로그인 SQLi 등)은 분석 대상에서 제외**됨을 인지할 것
>
> 이 셀을 건너뛰면 기존 데이터(전체 메서드, 중복 포함)로 진행됩니다.

> **⚠ 라벨 비율 변화 안내 (옵션 A 채택)**:
> GET 필터 + 중복 제거 후엔 라벨 비율이 약 **Normal 36% : Anomalous 64%** 로 뒤집힙니다 (정상 트래픽은 반복적이라 dedup으로 더 많이 압축되기 때문). 본 수업은 **데이터 다양성을 보존**하는 옵션 A를 채택하여 비율은 그대로 두고, 모델 학습 시 다음과 같이 보정합니다:
>
> - **RandomForest**: `class_weight='balanced'`
> - **XGBoost**: `scale_pos_weight = neg/pos` (자동 계산)
> - **LLM(Ollama)**: 보정 불필요 (제로샷 분류라 학습 데이터 분포에 무관)

In [2]:
# Step 0 (선택): GET만 + 중복 제거 + 10,000건 재샘플링
# ============================================================
import re

RANDOM_SEED = 42
TARGET_SIZE = 10_000

# 1) Kaggle 원본 다시 로드
KAGGLE_PATH = os.path.join(SCRIPT_DIR, "data", "archive", "csic_database.csv")
if not os.path.exists(KAGGLE_PATH):
    print(f"  !! Kaggle 원본을 찾을 수 없습니다: {KAGGLE_PATH}")
    print(f"  >> 이 셀은 건너뛰고 기존 df로 진행하세요.")
else:
    raw = pd.read_csv(KAGGLE_PATH)
    print(f"  Kaggle 원본 로드: {len(raw):,}건")

    # 2) download_csic2010.py와 동일한 컬럼 정규화
    URL_PREFIX_RE = re.compile(r"^https?://[^/]+", re.IGNORECASE)
    URL_SUFFIX_RE = re.compile(r"\s+HTTP/[\d.]+\s*$", re.IGNORECASE)

    def normalize_url(raw_url):
        if pd.isna(raw_url):
            return "", ""
        s = str(raw_url).strip()
        m = URL_SUFFIX_RE.search(s)
        http_version = m.group(0).strip() if m else "HTTP/1.1"
        s = URL_SUFFIX_RE.sub("", s)
        s = URL_PREFIX_RE.sub("", s)
        return s.strip(), http_version

    df_full = pd.DataFrame()
    df_full["label"] = raw.iloc[:, 0]
    df_full["method"] = raw["Method"]
    url_split = raw["URL"].apply(normalize_url)
    df_full["url"] = url_split.apply(lambda x: x[0])
    df_full["http_version"] = url_split.apply(lambda x: x[1])
    df_full["host"] = raw["host"]
    df_full["user_agent"] = raw["User-Agent"]
    df_full["pragma"] = raw["Pragma"]
    df_full["cache_control"] = raw["Cache-Control"]
    df_full["accept"] = raw["Accept"]
    df_full["accept_encoding"] = raw["Accept-encoding"]
    df_full["accept_charset"] = raw["Accept-charset"]
    df_full["language"] = raw["language"]
    df_full["cookie"] = raw["cookie"]
    df_full["content_type"] = raw["content-type"]
    df_full["content_length"] = raw["lenght"]
    df_full["connection"] = raw["connection"]
    df_full["body"] = raw["content"]
    df_full["classification"] = raw["classification"]

    # 3) ★ GET 요청만 필터링
    print()
    print(f"  메서드 분포 (필터링 전):")
    for m, c in df_full["method"].value_counts().items():
        print(f"    {m:6s}: {c:>7,}건")
    df_get = df_full[df_full["method"] == "GET"].reset_index(drop=True)
    print()
    print(f"  GET만 필터링: {len(df_full):,} → {len(df_get):,}건")

    # 4) 중복 제거 (method, url, body 기준)
    before = len(df_get)
    df_unique = df_get.drop_duplicates(subset=["method", "url", "body"]).reset_index(drop=True)
    removed = before - len(df_unique)
    print(f"  중복 제거: {before:,} → {len(df_unique):,}건 ({removed:,}건 제거)")
    print(f"    Normal:    {(df_unique['label']=='Normal').sum():,}건")
    print(f"    Anomalous: {(df_unique['label']=='Anomalous').sum():,}건")

    # 5) 라벨 비율 유지하며 최대 10,000건 재샘플링
    target = min(TARGET_SIZE, len(df_unique))
    if target < TARGET_SIZE:
        print()
        print(f"  !! 유니크 GET이 {len(df_unique):,}건뿐 → 전체 사용 ({TARGET_SIZE:,}건 미만)")

    ratios = df_unique["label"].value_counts(normalize=True)
    samples = []
    for label, ratio in ratios.items():
        n = int(target * ratio)
        pool = df_unique[df_unique["label"] == label]
        samples.append(pool.sample(n=min(n, len(pool)), random_state=RANDOM_SEED))

    df_resampled = pd.concat(samples).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

    # 6) 검증
    n_dup = df_resampled.duplicated(subset=["method", "url", "body"]).sum()
    methods = df_resampled["method"].unique().tolist()
    print()
    print(f"  최종: {len(df_resampled):,}건 (중복 {n_dup}건, 메서드 {methods})")
    print(f"  라벨 비율:")
    for label, count in df_resampled["label"].value_counts().items():
        pct = count / len(df_resampled) * 100
        bar = "█" * int(pct / 2)
        print(f"    {label:12s}: {count:>6,}건 ({pct:5.1f}%) {bar}")

    # 7) df 교체 + 저장
    df = df_resampled
    output_path = os.path.join(SCRIPT_DIR, "csic2010_get_unique.csv")
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print()
    print(f"  저장 완료: csic2010_get_unique.csv")
    print(f"  >> df가 GET 유니크 버전으로 교체되었습니다. 이후 셀은 이 데이터로 진행합니다.")

  Kaggle 원본 로드: 61,065건

  메서드 분포 (필터링 전):
    GET   :  43,088건
    POST  :  17,580건
    PUT   :     397건

  GET만 필터링: 61,065 → 43,088건
  중복 제거: 43,088 → 13,480건 (29,608건 제거)
    Normal:    4,832건
    Anomalous: 8,648건

  최종: 9,999건 (중복 0건, 메서드 ['GET'])
  라벨 비율:
    Anomalous   :  6,415건 ( 64.2%) ████████████████████████████████
    Normal      :  3,584건 ( 35.8%) █████████████████

  저장 완료: csic2010_get_unique.csv
  >> df가 GET 유니크 버전으로 교체되었습니다. 이후 셀은 이 데이터로 진행합니다.


## Step 1: URL 디코딩

In [3]:
print(f"\n{'─' * 65}")
print("  Step 1: URL 디코딩 (인코딩된 문자 → 원래 문자)")
print(f"{'─' * 65}")

df["url_decoded"] = df["url"].apply(
    lambda x: unquote(str(x), encoding="latin-1")
)
df["body_decoded"] = df["body"].fillna("").apply(
    lambda x: unquote(str(x), encoding="latin-1")
)

# 디코딩 전후 비교 예시
sample = df[df["url"].str.contains("%27|%3C|%2e", case=False, na=False)]
if len(sample) > 0:
    row = sample.iloc[0]
    print(f"\n  디코딩 전: {str(row['url'])[:80]}...")
    print(f"  디코딩 후: {str(row['url_decoded'])[:80]}...")
else:
    print(f"\n  디코딩 적용 완료")

# URL + Body 결합 텍스트
df["full_text"] = df["url_decoded"] + " " + df["body_decoded"]
print(f"\n  >> URL과 Body를 결합한 분석용 텍스트(full_text) 생성 완료")


─────────────────────────────────────────────────────────────────
  Step 1: URL 디코딩 (인코딩된 문자 → 원래 문자)
─────────────────────────────────────────────────────────────────

  디코딩 전: /tienda1/publico/autenticar.jsp?modo=entrar&login=yi%27j%2Be%2Can&pwd=2E9tifI6aR...
  디코딩 후: /tienda1/publico/autenticar.jsp?modo=entrar&login=yi'j+e,an&pwd=2E9tifI6aR&remem...

  >> URL과 Body를 결합한 분석용 텍스트(full_text) 생성 완료


## Step 2: 중복 확인

In [4]:
print(f"\n{'─' * 65}")
print("  Step 2: 중복 확인")
print(f"{'─' * 65}")

dup_count = df.duplicated(subset=["method", "url", "body"]).sum()
print(f"\n  중복 요청: {dup_count:,}건")
if dup_count > 0:
    print(f"  >> 학습용으로 중복은 유지합니다 (동일 요청 패턴도 의미 있음)")


─────────────────────────────────────────────────────────────────
  Step 2: 중복 확인
─────────────────────────────────────────────────────────────────

  중복 요청: 0건


## Step 3: 기본 길이 특성 추출

In [5]:
print(f"\n{'─' * 65}")
print("  Step 3: 기본 길이 특성 추출")
print(f"{'─' * 65}")

df["url_length"] = df["url_decoded"].str.len()
df["body_length"] = df["body_decoded"].str.len()
df["total_length"] = df["url_length"] + df["body_length"]

print(f"\n  {'특성':<15s} | {'정상 평균':>10s} | {'공격 평균':>10s} | {'차이':>8s}")
print(f"  {'─' * 15}-+-{'─' * 10}-+-{'─' * 10}-+-{'─' * 8}")

for feat in ["url_length", "body_length", "total_length"]:
    normal_mean = df[df["label"] == "Normal"][feat].mean()
    attack_mean = df[df["label"] == "Anomalous"][feat].mean()
    diff = attack_mean - normal_mean
    sign = "+" if diff > 0 else ""
    print(f"  {feat:<15s} | {normal_mean:>10.1f} | {attack_mean:>10.1f} | {sign}{diff:>7.1f}")


─────────────────────────────────────────────────────────────────
  Step 3: 기본 길이 특성 추출
─────────────────────────────────────────────────────────────────

  특성              |      정상 평균 |      공격 평균 |       차이
  ───────────────-+-──────────-+-──────────-+-────────
  url_length      |      163.1 |      145.0 |   -18.1
  body_length     |        0.0 |        0.0 |     0.0
  total_length    |      163.1 |      145.0 |   -18.1


## Step 4: URL 구조 특성 추출

In [6]:
print(f"\n{'─' * 65}")
print("  Step 4: URL 구조 특성 추출")
print(f"{'─' * 65}")


def extract_url_structure(url):
    """URL에서 구조적 특성 추출"""
    try:
        parsed = urlparse(url)
        params = parse_qs(parsed.query)
        return {
            "num_params": len(params),
            "path_depth": parsed.path.count("/"),
            "has_query": 1 if parsed.query else 0,
            "query_length": len(parsed.query),
        }
    except Exception:
        return {"num_params": 0, "path_depth": 0, "has_query": 0, "query_length": 0}


url_features = df["url_decoded"].apply(extract_url_structure).apply(pd.Series)
df = pd.concat([df, url_features], axis=1)

# POST body에서도 파라미터 수 추출
df["body_num_params"] = df["body_decoded"].apply(
    lambda x: len(x.split("&")) if x and "=" in x else 0
)

print(f"\n  추출된 URL 구조 특성: num_params, path_depth, has_query, query_length, body_num_params")
print(f"  >> 공격 요청의 파라미터 수 평균: {df[df['label']=='Anomalous']['num_params'].mean():.1f}")
print(f"  >> 정상 요청의 파라미터 수 평균: {df[df['label']=='Normal']['num_params'].mean():.1f}")


─────────────────────────────────────────────────────────────────
  Step 4: URL 구조 특성 추출
─────────────────────────────────────────────────────────────────

  추출된 URL 구조 특성: num_params, path_depth, has_query, query_length, body_num_params
  >> 공격 요청의 파라미터 수 평균: 6.0
  >> 정상 요청의 파라미터 수 평균: 7.9


## Step 5: 공격 키워드 탐지 특성 추출 ★★

In [7]:
print(f"\n{'─' * 65}")
print("  Step 5: 공격 키워드 탐지 특성 추출 ★")
print(f"{'─' * 65}")


def extract_attack_features(text):
    """텍스트에서 공격 관련 특성 추출"""
    text_lower = text.lower()

    # SQL Injection
    sql_keywords = ["select", "union", "drop", "insert", "delete",
                    "update", "' or", "1=1", "1'='1", "--", "/*"]
    has_sql = int(any(kw in text_lower for kw in sql_keywords))
    sql_count = sum(1 for kw in sql_keywords if kw in text_lower)

    # XSS
    xss_keywords = ["<script", "alert(", "onerror", "<iframe",
                    "<img", "javascript:", "onfocus", "onload"]
    has_xss = int(any(kw in text_lower for kw in xss_keywords))

    # Path Traversal
    has_traversal = int("../" in text or "..\\" in text)
    traversal_count = text.count("../") + text.count("..\\")

    # Command Injection ('|'는 정상 요청에도 나타날 수 있어 오탐 가능)
    cmd_patterns = ["; ", "| ", "&&", "/bin/", "/etc/", "exec"]
    has_cmd = int(any(p in text_lower for p in cmd_patterns))

    # CRLF Injection
    has_crlf = int("%0d" in text_lower or "%0a" in text_lower)

    # 특수문자 개수 및 비율
    num_special = len(re.findall(r"['\";|<>&`\\]", text))
    special_ratio = num_special / max(len(text), 1)

    # 숫자 점(dot) 슬래시 개수
    num_dots = text.count(".")
    num_slashes = text.count("/")

    # 공백 개수 (SQL Injection은 키워드 사이에 공백이 많음)
    num_spaces = text.count(" ") + text.count("%20") + text.count("+")

    return {
        "has_sql": has_sql, "sql_keyword_count": sql_count,
        "has_xss": has_xss, "has_traversal": has_traversal,
        "traversal_count": traversal_count,
        "has_cmd_injection": has_cmd, "has_crlf": has_crlf,
        "num_special_chars": num_special, "special_char_ratio": special_ratio,
        "num_dots": num_dots, "num_slashes": num_slashes,
        "num_spaces": num_spaces,
    }


print("\n  키워드 특성 추출 중...")
attack_features = df["full_text"].apply(extract_attack_features).apply(pd.Series)
df = pd.concat([df, attack_features], axis=1)

# 키워드 특성 요약
keyword_features = ["has_sql", "has_xss", "has_traversal", "has_cmd_injection", "has_crlf"]
print(f"\n  {'특성':<20s} | {'정상':>6s} | {'공격':>6s} | 의미")
print(f"  {'─' * 20}-+-{'─' * 6}-+-{'─' * 6}-+-{'─' * 25}")

for feat in keyword_features:
    normal_sum = df[df["label"] == "Normal"][feat].sum()
    attack_sum = df[df["label"] == "Anomalous"][feat].sum()
    print(f"  {feat:<20s} | {normal_sum:>6,} | {attack_sum:>6,} | "
          f"{'공격에 집중!' if attack_sum > normal_sum * 2 else '양쪽 모두 존재'}")


─────────────────────────────────────────────────────────────────
  Step 5: 공격 키워드 탐지 특성 추출 ★
─────────────────────────────────────────────────────────────────

  키워드 특성 추출 중...

  특성                   |     정상 |     공격 | 의미
  ────────────────────-+-──────-+-──────-+-─────────────────────────
  has_sql              |  713.0 | 1,490.0 | 공격에 집중!
  has_xss              |    0.0 |  172.0 | 공격에 집중!
  has_traversal        |    0.0 |    0.0 | 양쪽 모두 존재
  has_cmd_injection    |    0.0 |  151.0 | 공격에 집중!
  has_crlf             |    0.0 |  240.0 | 공격에 집중!


## Step 6: 통계적 특성 추출

In [8]:
print(f"\n{'─' * 65}")
print("  Step 6: 통계적 특성 추출 (엔트로피, 비율)")
print(f"{'─' * 65}")


def calculate_entropy(text):
    """섀넌 엔트로피 계산"""
    if not text:
        return 0.0
    freq = {}
    for char in text:
        freq[char] = freq.get(char, 0) + 1
    length = len(text)
    return -sum((c / length) * math.log2(c / length) for c in freq.values())


print("\n  엔트로피 계산 중...")
df["url_entropy"] = df["url_decoded"].apply(calculate_entropy)

df["digit_ratio"] = df["full_text"].apply(
    lambda x: sum(c.isdigit() for c in x) / max(len(x), 1)
)
df["upper_ratio"] = df["full_text"].apply(
    lambda x: sum(c.isupper() for c in x) / max(len(x), 1)
)

print(f"\n  정상 URL 엔트로피 평균: {df[df['label']=='Normal']['url_entropy'].mean():.3f}")
print(f"  공격 URL 엔트로피 평균: {df[df['label']=='Anomalous']['url_entropy'].mean():.3f}")
print(f"  >> 공격 요청은 특수문자가 많아 엔트로피가 다른 경향이 있습니다")


─────────────────────────────────────────────────────────────────
  Step 6: 통계적 특성 추출 (엔트로피, 비율)
─────────────────────────────────────────────────────────────────

  엔트로피 계산 중...

  정상 URL 엔트로피 평균: 4.759
  공격 URL 엔트로피 평균: 4.694
  >> 공격 요청은 특수문자가 많아 엔트로피가 다른 경향이 있습니다


## Step 6.5 (참고): 공격 유형별 엔트로피 분석

Step 6에서 정상/공격 평균 엔트로피 차이가 작게 나온 이유는 **공격 유형마다 엔트로피가 반대 방향으로 움직이기 때문**입니다:

- **XSS / SQL Injection**: 다양한 특수문자(`<`, `>`, `'`, `"`, `=`, ...) → 엔트로피 ↑
- **Path Traversal**: `../../../` 같은 패턴 반복 → 엔트로피 ↓
- **Buffer Overflow**: `AAAAAA...` → 엔트로피 ↓↓

평균만 보면 양방향 효과가 상쇄되어 작아 보이지만, **분산이 커서** 트리 모델에선 여전히 유용한 특성입니다.

In [9]:
print()
print("  " + "─" * 65)
print("  Step 6.5: 공격 유형별 엔트로피 분석 (참고)")
print("  " + "─" * 65)

def classify_attack_type(text):
    """공격 텍스트를 키워드 기반으로 대분류"""
    t = str(text).lower()
    if "../" in t or "%2e%2e" in t:
        return "Path Traversal"
    if "<script" in t or "alert(" in t or "<iframe" in t or "javascript:" in t:
        return "XSS"
    if "' or" in t or "1=1" in t or "union" in t or " select " in t or "--" in t:
        return "SQL Injection"
    if "%0d" in t or "%0a" in t:
        return "CRLF Injection"
    if "/etc/" in t or "/bin/" in t or "exec" in t:
        return "Command Injection"
    return "Other"

# 공격 데이터에 분류 적용
attack_df = df[df["label"] == "Anomalous"].copy()
attack_df["attack_type"] = attack_df["full_text"].apply(classify_attack_type)

# 정상 평균 (비교 기준선)
normal_entropy = df.loc[df["label"] == "Normal", "url_entropy"].mean()

# 공격 유형별 엔트로피 평균
grouped = attack_df.groupby("attack_type")["url_entropy"].agg(["mean", "count"])
grouped = grouped.sort_values("mean")

print()
print(f"  정상 평균 엔트로피: {normal_entropy:.3f}  (비교 기준선)")
print()
print(f"  {'공격 유형':<20s} | {'엔트로피':>10s} | {'정상 대비':>10s} | {'건수':>8s} | 방향")
print("  " + "─" * 70)

for atype, row in grouped.iterrows():
    avg = row["mean"]
    cnt = int(row["count"])
    diff = avg - normal_entropy
    if diff > 0.1:
        arrow = "↑↑ 높음"
    elif diff > 0:
        arrow = "↑  약간 높음"
    elif diff > -0.1:
        arrow = "·  비슷함"
    else:
        arrow = "↓  낮음"
    sign = "+" if diff >= 0 else ""
    print(f"  {atype:<20s} | {avg:>10.3f} | {sign}{diff:>9.3f} | {cnt:>8,} | {arrow}")

print("  " + "─" * 70)
print()
print("  >> 공격 유형마다 엔트로피가 ↑↓ 반대 방향으로 움직임")
print("  >> 평균 차이가 작은 것은 양방향 효과가 상쇄되기 때문")
print("  >> 트리 모델은 분산을 활용하므로 여전히 유용한 특성!")


  ─────────────────────────────────────────────────────────────────
  Step 6.5: 공격 유형별 엔트로피 분석 (참고)
  ─────────────────────────────────────────────────────────────────

  정상 평균 엔트로피: 4.759  (비교 기준선)

  공격 유형                |       엔트로피 |      정상 대비 |       건수 | 방향
  ──────────────────────────────────────────────────────────────────────
  Other                |      4.640 |    -0.119 |    5,494 | ↓  낮음
  XSS                  |      4.973 | +    0.214 |      172 | ↑↑ 높음
  SQL Injection        |      4.982 | +    0.223 |      509 | ↑↑ 높음
  CRLF Injection       |      5.114 | +    0.355 |      240 | ↑↑ 높음
  ──────────────────────────────────────────────────────────────────────

  >> 공격 유형마다 엔트로피가 ↑↓ 반대 방향으로 움직임
  >> 평균 차이가 작은 것은 양방향 효과가 상쇄되기 때문
  >> 트리 모델은 분산을 활용하므로 여전히 유용한 특성!


## Step 7: 전처리 결과 저장

In [10]:
print(f"\n{'─' * 65}")
print("  Step 7: 전처리 결과 저장")
print(f"{'─' * 65}")

output_path = os.path.join(SCRIPT_DIR, "csic2010_cleaned.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")

# 추출된 특성 목록 출력
feature_cols = [
    "url_length", "body_length", "total_length",
    "num_params", "path_depth", "has_query", "query_length", "body_num_params",
    "has_sql", "sql_keyword_count", "has_xss",
    "has_traversal", "traversal_count",
    "has_cmd_injection", "has_crlf",
    "num_special_chars", "special_char_ratio",
    "num_dots", "num_slashes", "num_spaces",
    "url_entropy", "digit_ratio", "upper_ratio",
]

print(f"\n  저장 완료: csic2010_cleaned.csv")
print(f"  최종 데이터: {df.shape[0]:,}행 x {df.shape[1]}열")
print(f"\n  추출된 숫자형 특성 ({len(feature_cols)}개):")
for i, col in enumerate(feature_cols, 1):
    print(f"    {i:2d}. {col}")

print(f"\n{'=' * 65}")
print("  전처리 + 특성 추출 완료! 다음: feature_engineering.ipynb")
print(f"{'=' * 65}")


─────────────────────────────────────────────────────────────────
  Step 7: 전처리 결과 저장
─────────────────────────────────────────────────────────────────

  저장 완료: csic2010_cleaned.csv
  최종 데이터: 9,999행 x 44열

  추출된 숫자형 특성 (23개):
     1. url_length
     2. body_length
     3. total_length
     4. num_params
     5. path_depth
     6. has_query
     7. query_length
     8. body_num_params
     9. has_sql
    10. sql_keyword_count
    11. has_xss
    12. has_traversal
    13. traversal_count
    14. has_cmd_injection
    15. has_crlf
    16. num_special_chars
    17. special_char_ratio
    18. num_dots
    19. num_slashes
    20. num_spaces
    21. url_entropy
    22. digit_ratio
    23. upper_ratio

  전처리 + 특성 추출 완료! 다음: feature_engineering.ipynb
